<div dir="rtl">
  <p style="color: #80f9fa; text-align: right;">
    <b>بلوک residual</b>
  </p>
</div>

---

In [2]:
import torch
import torch.nn as nn


class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.relu(out)
        out = self.conv2(out)
        out += residual
        out = self.relu(out)
        return out


x = torch.randn(1, 3, 32, 32)
model = ResidualBlock(3)
y = model(x)
print(y.shape)

torch.Size([1, 3, 32, 32])


In [3]:
import torch
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, in_channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.conv2 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(in_channels)

    def forward(self, x):
        residual = x  # Skip connection
        out = self.conv1(x)
        out = self.bn1(out)
        out = torch.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += residual  # Adding the input (skip connection)
        out = torch.relu(out)
        return out

x = torch.randn(1, 3, 32, 32)
model = ResidualBlock(3)
y = model(x)
print(y.shape)

torch.Size([1, 3, 32, 32])


In [ ]:
import torch
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, in_channels):
        super(ResidualBlock, self).__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(in_channels)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(in_channels)
        )

    def forward(self, x):
        b1 = torch.relu(x + self.block1(x))
        print()
        b2 = torch.relu(b1 + self.block2(b1))
        return b2

model = ResidualBlock(3)
x = torch.randn(1, 3, 32, 32)
y = model(x)
print(y.shape)

<div dir="rtl" style="text-align: right;">
شبکه‌ی
resnet:
نمونه‌ای موفق از معماری 
residual
</div>

In [ ]:
import torchvision.models as models

# resnet18 = models.resnet18(pretrained=False)
resnet18 = models.resnet18(weights=None)

In [24]:
import torch
from torchviz import make_dot   # # https://graphviz.org/download/ 
from torchvision.models import resnet18

model = resnet18(weights=None)
x = torch.randn(1, 3, 224, 224)
y = model(x)

make_dot(y, params=dict(model.named_parameters())).render("../pictures/resnet18_graph", format="png")

'..\\pictures\\resnet18_graph.png'

---

<div dir="rtl" style="text-align: right;">
تمرین:
یک شبکه عصبی کانولوشنی برای داده‌ی
mnist
براساس معماری زیر و مطابق با قالب پیوست، آموزش بدهید.

معماری پیشنهادی

- بلوک Sequential:
    - Conv2D(32, 3×3, padding=1) → ReLU → MaxPool(2×2)

- (Residual Block):
    - Conv2D(32, 3×3, padding=1) → BatchNorm → ReLU
    - Conv2D(64, 3×3, padding=1) → BatchNorm → ReLU
    - Conv2D(32, 3×3, padding=1) → BatchNorm → ReLU

- لایه‌های Fully Connected (Dense) در مدل Sequential:
    - Flatten()
    - Dense(512) → ReLU → Dropout(0.4)
    - Dense(256) → ReLU → Dropout(0.4)
    - Dense(128) → ReLU
    - Dense(10)
</div>

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Model(nn.Module):
    def __init__(self, num_classes=10):
        super(Model, self).__init__()
        

        self.conv_block1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding="same"),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )


        self.conv_block2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding="same"),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding="same"),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=32, kernel_size=3, padding="same"),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )


        self.fc_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 14 * 14, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = x + self.conv_block2(x)
        x = self.fc_block(x)
        return x

x = torch.randn(1, 1, 28, 28)
model = Model()
y = model(x)
print(y.shape)

torch.Size([1, 10])
